# IndoBERT Fine-tuning — Skenario B-Gabungan
**Data:** Gabungan → Train:66.428 | Test:3.707 (fix)  
**Split:** 80:20 tanpa validation set — konsisten dengan LR dan XGBoost (sesuai saran dosen)  
**Referensi parameter:**  
- lr=2e-5, epoch=5: Braja et al. (2023)  
- batch=16: Zaidan et al. (2024)  
- max_length=128, warmup 10%: Sagama et al. (2023); Shaw et al. (2025)  
- class weight balanced: Sagama et al. (2023)


## Cell 1 — Setup & Konfigurasi

In [1]:
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
SPLIT_DIR  = os.path.join(BASE_DIR, 'splits')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULT_DIR, exist_ok=True)

SEED=42; LABEL_MAP={'keluhan':0,'saran':1,'pujian':2}
INV_MAP={v:k for k,v in LABEL_MAP.items()}; CLASS_NAMES=['keluhan','saran','pujian']

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('CPU mode')

# ── Konfigurasi IndoBERT (Referensi: Zaidan et al., 2024; Braja et al., 2023) ─
MODEL_NAME    = 'indobenchmark/indobert-base-p1'
MAX_LENGTH    = 128   # Braja et al. (2023); Shaw et al. (2025)
EPOCHS        = 5     # Braja et al. (2023): epoch 5 optimal
BATCH_SIZE    = 16    # Zaidan et al. (2024): batch 16 RTX GPU
LEARNING_RATE = 2e-5  # Braja et al. (2023); Sagama et al. (2023)
WARMUP_RATIO  = 0.1   # Sagama et al. (2023): 10% warmup linear scheduler

print(f'\nKonfigurasi IndoBERT:')
print(f'  Model      : {MODEL_NAME}')
print(f'  Max Length : {MAX_LENGTH}  [Braja et al., 2023]')
print(f'  Epochs     : {EPOCHS}      [Braja et al., 2023]')
print(f'  Batch Size : {BATCH_SIZE}  [Zaidan et al., 2024]')
print(f'  LR         : {LEARNING_RATE}  [Braja et al., 2023]')
print(f'  Warmup     : {int(WARMUP_RATIO*100)}%  [Sagama et al., 2023]')

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LENGTH):
        self.texts=list(texts); self.labels=list(labels)
        self.tokenizer=tokenizer; self.max_len=max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc=self.tokenizer(str(self.texts[idx]),max_length=self.max_len,
            padding='max_length',truncation=True,return_tensors='pt')
        return {'input_ids':enc['input_ids'].squeeze(),
                'attention_mask':enc['attention_mask'].squeeze(),
                'labels':torch.tensor(int(self.labels[idx]),dtype=torch.long)}

def evaluate_idb(model, loader, device):
    model.eval(); preds=[]; labels_all=[]
    with torch.no_grad():
        for batch in loader:
            out=model(input_ids=batch['input_ids'].to(device),
                      attention_mask=batch['attention_mask'].to(device))
            preds.extend(torch.argmax(out.logits,1).cpu().numpy())
            labels_all.extend(batch['labels'].numpy())
    return np.array(preds), np.array(labels_all)

def plot_eval(y_true, y_pred, exp_code, prefix):
    acc=accuracy_score(y_true,y_pred)
    mac=f1_score(y_true,y_pred,average='macro',zero_division=0)
    f1s=f1_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    prec=precision_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    rec=recall_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    rep=classification_report(y_true,y_pred,target_names=CLASS_NAMES,digits=4,zero_division=0)
    print(f'\n{"="*60}'); print(f'HASIL — {exp_code}'); print(f'{"="*60}')
    print(f'  Accuracy   : {acc*100:.2f}%')
    print(f'  Macro F1   : {mac:.4f}')
    print(f'  F1 Keluhan : {f1s[0]:.4f}  Prec:{prec[0]:.4f}  Rec:{rec[0]:.4f}')
    print(f'  F1 Saran   : {f1s[1]:.4f}  Prec:{prec[1]:.4f}  Rec:{rec[1]:.4f}')
    print(f'  F1 Pujian  : {f1s[2]:.4f}  Prec:{prec[2]:.4f}  Rec:{rec[2]:.4f}')
    print(f'\n{rep}')
    cm=confusion_matrix(y_true,y_pred,labels=[0,1,2])
    fig,ax=plt.subplots(figsize=(6,5)); vmax=cm.max()
    im=ax.imshow(cm,cmap='Blues',vmin=0,vmax=vmax)
    for i in range(3):
        for j in range(3):
            c='white' if cm[i,j]>vmax*0.55 else '#1A1A1A'
            ax.text(j,i,f'{cm[i,j]:,}',ha='center',va='center',fontsize=12,fontweight='bold',color=c)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks([0,1,2]); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicted',fontweight='bold'); ax.set_ylabel('Actual',fontweight='bold')
    ax.set_title(f'{exp_code}\nAcc={acc*100:.2f}% | MacroF1={mac:.4f}',fontweight='bold',pad=10)
    plt.colorbar(im,ax=ax,shrink=0.85); fig.tight_layout()
    path=os.path.join(RESULT_DIR,f'CM_{prefix}.png')
    fig.savefig(path); plt.close(); print(f'  CM → {path}')
    return {'exp':exp_code,'accuracy':acc,'macro_f1':mac,
            'f1_keluhan':f1s[0],'f1_saran':f1s[1],'f1_pujian':f1s[2]}

print('Setup selesai!')


GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB

Konfigurasi IndoBERT:
  Model      : indobenchmark/indobert-base-p1
  Max Length : 128  [Braja et al., 2023]
  Epochs     : 5      [Braja et al., 2023]
  Batch Size : 16  [Zaidan et al., 2024]
  LR         : 2e-05  [Braja et al., 2023]
  Warmup     : 10%  [Sagama et al., 2023]
Setup selesai!


## Cell 2 — Load Data Split

In [2]:
df_train=pd.read_csv(os.path.join(SPLIT_DIR,'B_gabungan_idb_train.csv'))
df_test =pd.read_csv(os.path.join(SPLIT_DIR,'B_gabungan_idb_test.csv'))
df_train['label_enc']=df_train['label_pks'].map(LABEL_MAP)
df_test['label_enc'] =df_test['label_pks'].map(LABEL_MAP)

X_train=df_train['text'].fillna('').values; y_train=df_train['label_enc'].values
X_test =df_test['text'].fillna('').values;  y_test =df_test['label_enc'].values

print(f'Skenario : B-Gabungan')
print(f'Train    : {len(X_train):,} baris')
print(f'Test     : {len(X_test):,} baris')
# TIDAK ADA validation set — sesuai saran dosen (apple-to-apple dengan LR & XGB)
print('Split: Train 80% | Test 20% (tanpa validation — konsisten dengan LR & XGB)')


Skenario : B-Gabungan
Train    : 54,142 baris
Test     : 2,999 baris
Split: Train 80% | Test 20% (tanpa validation — konsisten dengan LR & XGB)


## Cell 3 — Load Tokenizer & Model

In [3]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

config=AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels=3
config.id2label=INV_MAP
config.label2id=LABEL_MAP

# Load dari pretrained base (BUKAN dari model pelabel)
# Model pelabel (indobert_labeler) hanya digunakan untuk pseudo-labeling data
# Model klasifikasi ini di-fine-tune ulang dari indobert-base-p1
model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=config, ignore_mismatched_sizes=True
).to(DEVICE)

print(f'Model: {MODEL_NAME}')
print(f'Labels: {model.config.id2label}')
print(f'Device: {DEVICE}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: indobenchmark/indobert-base-p1
Labels: {0: 'keluhan', 1: 'saran', 2: 'pujian'}
Device: cuda


## Cell 4 — Dataset & DataLoader

In [4]:
train_ds=SentimentDataset(X_train,y_train,tokenizer)
test_ds =SentimentDataset(X_test, y_test, tokenizer)

train_dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,
                    num_workers=0,pin_memory=True)
test_dl =DataLoader(test_ds, batch_size=32,         shuffle=False,
                    num_workers=0,pin_memory=True)

print(f'Train batches: {len(train_dl)}')
print(f'Test  batches: {len(test_dl)}')

Train batches: 3384
Test  batches: 94


## Cell 5 — Optimizer, Loss, Scheduler

In [5]:
# Class weight untuk menangani imbalance
# Referensi: Sagama et al. (2023)
cw=compute_class_weight('balanced',classes=np.array([0,1,2]),y=y_train)
cw_tensor=torch.tensor(cw,dtype=torch.float).to(DEVICE)
loss_fn=torch.nn.CrossEntropyLoss(weight=cw_tensor)

# AdamW + weight decay: Sagama et al. (2023)
optimizer=AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=0.01)

# Linear warmup scheduler 10%: Shaw et al. (2025)
total_steps=len(train_dl)*EPOCHS
warmup_steps=int(total_steps*WARMUP_RATIO)
scheduler=get_linear_schedule_with_warmup(optimizer,warmup_steps,total_steps)

print(f'Total steps  : {total_steps}')
print(f'Warmup steps : {warmup_steps}')
print(f'Class weight : keluhan={cw[0]:.3f} saran={cw[1]:.3f} pujian={cw[2]:.3f}')

Total steps  : 16920
Warmup steps : 1692
Class weight : keluhan=0.475 saran=2.298 pujian=2.175


## Cell 6 — Fine-tuning IndoBERT
> **Catatan metodologi:** Tidak ada validation set — evaluasi dilakukan pada test set per epoch  
> Konsisten dengan LR & XGBoost yang langsung mengevaluasi pada test set (sesuai saran dosen)  
> **Estimasi waktu:** ~2-3 jam


In [6]:
SAVE_DIR=os.path.join(BASE_DIR,'indobert_B_Gabungan')
os.makedirs(SAVE_DIR,exist_ok=True)

history=[]; best_f1=0.0
print(f'Fine-tuning IndoBERT — Skenario B-Gabungan')
print('='*60)

for epoch in range(EPOCHS):
    t0=time.time()
    # ── Training ────────────────────────────────────────────────────────────
    model.train(); total_loss=0
    for batch in train_dl:
        ids =batch['input_ids'].to(DEVICE)
        mask=batch['attention_mask'].to(DEVICE)
        lbl =batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        out =model(input_ids=ids,attention_mask=mask)
        loss=loss_fn(out.logits,lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step(); scheduler.step()
        total_loss+=loss.item()
    avg_loss=total_loss/len(train_dl)

    # ── Evaluasi pada TEST SET per epoch (konsisten dengan XGBoost/LR) ─────
    preds_ep, labels_ep = evaluate_idb(model, test_dl, DEVICE)
    ep_acc=accuracy_score(labels_ep,preds_ep)
    ep_f1 =f1_score(labels_ep,preds_ep,average='macro',zero_division=0)

    elapsed=(time.time()-t0)/60
    history.append({'epoch':epoch+1,'loss':avg_loss,'acc':ep_acc,'f1':ep_f1})
    print(f'Epoch {epoch+1}/{EPOCHS} ({elapsed:.1f} min) '
          f'Loss:{avg_loss:.4f} Acc:{ep_acc*100:.2f}% F1:{ep_f1:.4f}')

    # ── Simpan model terbaik ────────────────────────────────────────────────
    if ep_f1>best_f1:
        best_f1=ep_f1
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f'  Best model saved (F1={best_f1:.4f})')

print(f'\nFine-tuning selesai! Best F1: {best_f1:.4f}')

Fine-tuning IndoBERT — Skenario B-Gabungan
Epoch 1/5 (11.5 min) Loss:0.3471 Acc:94.20% F1:0.9185


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Best model saved (F1=0.9185)
Epoch 2/5 (11.6 min) Loss:0.2259 Acc:95.20% F1:0.9303


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Best model saved (F1=0.9303)
Epoch 3/5 (11.6 min) Loss:0.1465 Acc:94.90% F1:0.9283
Epoch 4/5 (11.5 min) Loss:0.0909 Acc:95.27% F1:0.9314


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Best model saved (F1=0.9314)
Epoch 5/5 (11.6 min) Loss:0.0479 Acc:95.37% F1:0.9331


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Best model saved (F1=0.9331)

Fine-tuning selesai! Best F1: 0.9331


## Cell 7 — Evaluasi Final (Best Model) + Training History

In [7]:
# Load model terbaik dan evaluasi final
best_model=AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(DEVICE)
preds_final, labels_final = evaluate_idb(best_model, test_dl, DEVICE)
result=plot_eval(labels_final,preds_final,'IndoBERT-B-Gabungan','IndoBERT_B_Gabungan')

# Plot training history
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4))
ep_x=[h['epoch'] for h in history]
ax1.plot(ep_x,[h['loss'] for h in history],'o-',color='#C04040',lw=2,ms=7)
ax1.set_title('Training Loss per Epoch',fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(True,alpha=0.3)
ax2.plot(ep_x,[h['acc']*100 for h in history],'o-',color='#2471A3',lw=2,ms=7,label='Accuracy (%)')
ax2.plot(ep_x,[h['f1'] for h in history],'s--',color='#1D9E75',lw=2,ms=7,label='Macro F1')
ax2.set_title('Test Performance per Epoch — B-Gabungan',fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True,alpha=0.3)
plt.tight_layout()
hist_path=os.path.join(RESULT_DIR,'hist_IndoBERT_B_Gabungan.png')
plt.savefig(hist_path); plt.close(); print(f'History → {hist_path}')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


HASIL — IndoBERT-B-Gabungan
  Accuracy   : 95.37%
  Macro F1   : 0.9331
  F1 Keluhan : 0.9699  Prec:0.9687  Rec:0.9710
  F1 Saran   : 0.8861  Prec:0.8871  Rec:0.8851
  F1 Pujian  : 0.9432  Prec:0.9474  Rec:0.9391

              precision    recall  f1-score   support

     keluhan     0.9687    0.9710    0.9699      2104
       saran     0.8871    0.8851    0.8861       435
      pujian     0.9474    0.9391    0.9432       460

    accuracy                         0.9537      2999
   macro avg     0.9344    0.9317    0.9331      2999
weighted avg     0.9536    0.9537    0.9536      2999

  CM → C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\results\CM_IndoBERT_B_Gabungan.png
History → C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\results\hist_IndoBERT_B_Gabungan.png
